In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

from datetime import datetime

print("Libraries imported successfully")

Libraries imported successfully


In [0]:
# COMMAND ----------

CATALOG = "retail_ai_prod"
GOLD_SCHEMA = "gold_data"

VOLUME_PATH = (
    "/Volumes/retail_ai_prod/"
    "unstructured_docs/retail_documents"
)

SOURCE_PATH = f"{VOLUME_PATH}/landing/structured_data"

SALES_PATH = f"{SOURCE_PATH}/sales_transactions.csv"
INVENTORY_PATH = f"{SOURCE_PATH}/inventory_levels.csv"
CUSTOMERS_PATH = f"{SOURCE_PATH}/customer_profiles.csv"

SALES_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.sales_transactions"
INVENTORY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.inventory_levels"
CUSTOMER_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.customer_profiles"

print("Source:", SOURCE_PATH)
print("Destination:", CATALOG, GOLD_SCHEMA)

Source: /Volumes/retail_ai_prod/unstructured_docs/retail_documents/landing/structured_data
Destination: retail_ai_prod gold_data


In [0]:
# COMMAND ----------

sales_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("transaction_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("store_city", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("gross_sales", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("returns_amount", DoubleType(), True),
    StructField("net_revenue", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("channel", StringType(), True)
])

print("Sales schema defined")

Sales schema defined


In [0]:
# COMMAND ----------

inventory_schema = StructType([
    StructField("snapshot_date", StringType(), True),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("store_city", StringType(), True),
    StructField("on_hand_qty", IntegerType(), True),
    StructField("reserved_qty", IntegerType(), True),
    StructField("reorder_level", IntegerType(), True),
    StructField("unit_cost", DoubleType(), True),
    StructField("supplier_id", StringType(), True)
])

print("Inventory schema defined")

Inventory schema defined


In [0]:
# COMMAND ----------

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("preferred_channel", StringType(), True),
    StructField("age_band", StringType(), True)
])

print("Customer schema defined")

Customer schema defined


#Reading

In [0]:
# COMMAND ----------

sales_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", ",")
    .option("mode", "FAILFAST")
    .schema(sales_schema)
    .load(SALES_PATH)
)

display(sales_df)

transaction_id,transaction_date,customer_id,product_id,product_name,category,store_city,quantity,gross_sales,discount_amount,returns_amount,net_revenue,payment_method,channel
T000001,2026-03-27,C0026,P1003,Smart Watch,Electronics,Kolkata,3,17997.0,0.0,17997.0,0.0,UPI,Online
T000002,2025-09-29,C0087,P3002,Running Shoes,Fashion,Ahmedabad,2,5998.0,0.0,0.0,5998.0,UPI,Store
T000003,2026-05-15,C0139,P6001,Yoga Mat,Sports,Jaipur,4,3996.0,399.6,1998.0,1598.4,UPI,Online
T000004,2025-04-03,C0112,P6001,Yoga Mat,Sports,Delhi,3,2997.0,149.85,1498.5,1348.65,UPI,Store
T000005,2025-11-29,C0015,P1003,Smart Watch,Electronics,Hyderabad,3,17997.0,0.0,0.0,17997.0,Cash,Store
T000006,2026-02-26,C0145,P3002,Running Shoes,Fashion,Mumbai,2,5998.0,299.9,0.0,5698.1,Card,Online
T000007,2026-08-21,C0037,P1002,Bluetooth Speaker,Electronics,Bengaluru,4,13996.0,699.8,0.0,13296.2,Wallet,Store
T000008,2026-04-22,C0074,P4001,Face Serum,Beauty,Mumbai,3,2697.0,0.0,2697.0,0.0,UPI,Store
T000009,2025-09-14,C0117,P3001,Cotton T-Shirt,Fashion,Hyderabad,3,2397.0,359.55,1198.5,838.95,Card,Store
T000010,2026-08-09,C0092,P4001,Face Serum,Beauty,Hyderabad,1,899.0,44.95,0.0,854.05,Cash,Online


In [0]:
# COMMAND ----------

inventory_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(inventory_schema)
    .load(INVENTORY_PATH)
)

display(inventory_df)

snapshot_date,product_id,product_name,category,store_city,on_hand_qty,reserved_qty,reorder_level,unit_cost,supplier_id
2026-11-15,P1001,Wireless Earbuds,Electronics,Jaipur,155,2,29,1545.96,V004
2026-11-15,P1001,Wireless Earbuds,Electronics,Delhi,44,5,17,1648.97,V006
2026-11-15,P1001,Wireless Earbuds,Electronics,Mumbai,13,7,28,1361.39,V006
2026-11-15,P1001,Wireless Earbuds,Electronics,Bengaluru,28,10,25,1280.59,V002
2026-11-15,P1001,Wireless Earbuds,Electronics,Hyderabad,74,7,24,1307.94,V004
2026-11-15,P1001,Wireless Earbuds,Electronics,Pune,0,0,38,1260.41,V005
2026-11-15,P1001,Wireless Earbuds,Electronics,Ahmedabad,164,0,31,1708.95,V008
2026-11-15,P1001,Wireless Earbuds,Electronics,Kolkata,14,0,26,1509.1,V006
2026-11-15,P1002,Bluetooth Speaker,Electronics,Jaipur,76,1,26,1579.06,V003
2026-11-15,P1002,Bluetooth Speaker,Electronics,Delhi,102,0,26,1663.2,V008


In [0]:
# COMMAND ----------

customer_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(customer_schema)
    .load(CUSTOMERS_PATH)
)

display(customer_df)

customer_id,customer_name,city,signup_date,loyalty_tier,preferred_channel,age_band
C0001,Customer 001,Delhi,2022-02-21,Gold,App,26-35
C0002,Customer 002,Mumbai,2022-07-29,Bronze,Store,46-60
C0003,Customer 003,Jaipur,2022-03-03,Bronze,App,26-35
C0004,Customer 004,Jaipur,2025-02-23,Silver,Store,60+
C0005,Customer 005,Ahmedabad,2023-03-28,Platinum,Store,36-45
C0006,Customer 006,Jaipur,2022-11-23,Platinum,Web,36-45
C0007,Customer 007,Mumbai,2023-03-17,Gold,App,18-25
C0008,Customer 008,Ahmedabad,2022-07-18,Gold,Web,60+
C0009,Customer 009,Hyderabad,2022-03-30,Platinum,Store,18-25
C0010,Customer 010,Ahmedabad,2022-06-11,Gold,Store,60+


#Velidating

In [0]:
# COMMAND ----------

sales_df = sales_df.withColumn(
    "transaction_date",
    F.to_date("transaction_date", "yyyy-MM-dd")
)

inventory_df = inventory_df.withColumn(
    "snapshot_date",
    F.to_date("snapshot_date", "yyyy-MM-dd")
)

customer_df = customer_df.withColumn(
    "signup_date",
    F.to_date("signup_date", "yyyy-MM-dd")
)

print("Date conversion completed")

Date conversion completed


In [0]:
# COMMAND ----------

sales_count = sales_df.count()
inventory_count = inventory_df.count()
customer_count = customer_df.count()

print("Sales records:", sales_count)
print("Inventory records:", inventory_count)
print("Customer records:", customer_count)

Sales records: 1000
Inventory records: 96
Customer records: 200


##Writing into tables

In [0]:
# COMMAND ----------

(
    sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SALES_TABLE)
)

print("Sales Delta table created!")

Sales Delta table created!


In [0]:
# COMMAND ----------

(
    inventory_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(INVENTORY_TABLE)
)

print("Inventory Delta table created!")

Inventory Delta table created!


In [0]:
# COMMAND ----------

(
    customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CUSTOMER_TABLE)
)

print("Customer Delta table created!")

Customer Delta table created!


In [0]:
%sql
-- COMMAND ----------

COMMENT ON TABLE
retail_ai_prod.gold_data.sales_transactions
IS 'Curated retail sales transaction data containing gross sales, discounts, returns and net revenue';

COMMENT ON TABLE
retail_ai_prod.gold_data.inventory_levels
IS 'Curated inventory snapshots by product and store city, including available stock and reorder levels';

COMMENT ON TABLE
retail_ai_prod.gold_data.customer_profiles
IS 'Curated customer master data containing location, loyalty tier and channel preferences';

##Verification

In [0]:
%sql
-- COMMAND ----------

SELECT
    store_city,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY store_city
ORDER BY total_revenue DESC;

store_city,total_revenue
Jaipur,670986.7
Kolkata,654423.3
Ahmedabad,624024.05
Pune,584816.75
Hyderabad,570575.9
Mumbai,542977.1
Delhi,504031.4
Bengaluru,428752.95


In [0]:
%sql
-- COMMAND ----------

SELECT
    product_id,
    product_name,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY product_id, product_name
ORDER BY total_revenue DESC
LIMIT 10;

product_id,product_name,total_quantity_sold,total_revenue
P2002,Air Fryer,256,1071685.1
P1003,Smart Watch,219,952941.15
P2001,Mixer Grinder,229,517283.2
P3002,Running Shoes,247,465594.75
P1002,Bluetooth Speaker,175,462042.95
P1001,Wireless Earbuds,175,288384.6
P6002,Dumbbell Set,190,281142.15
P6001,Yoga Mat,204,138461.4
P4001,Face Serum,218,127882.75
P5001,Organic Rice 5kg,196,101389.95


In [0]:
%sql
-- COMMAND ----------

SELECT
    product_id,
    product_name,
    store_city,
    on_hand_qty,
    reserved_qty,
    reorder_level
FROM retail_ai_prod.gold_data.inventory_levels
WHERE on_hand_qty - reserved_qty < reorder_level
ORDER BY store_city;

product_id,product_name,store_city,on_hand_qty,reserved_qty,reorder_level
P1002,Bluetooth Speaker,Ahmedabad,27,10,29
P6002,Dumbbell Set,Ahmedabad,19,0,40
P1003,Smart Watch,Bengaluru,30,7,28
P3001,Cotton T-Shirt,Bengaluru,22,1,43
P6001,Yoga Mat,Bengaluru,5,0,41
P1001,Wireless Earbuds,Bengaluru,28,10,25
P5001,Organic Rice 5kg,Delhi,45,6,41
P3001,Cotton T-Shirt,Hyderabad,10,10,33
P5001,Organic Rice 5kg,Hyderabad,3,0,34
P6001,Yoga Mat,Hyderabad,26,6,31


In [0]:
%sql
-- COMMAND ----------

SELECT
    loyalty_tier,
    COUNT(*) AS total_customers
FROM retail_ai_prod.gold_data.customer_profiles
GROUP BY loyalty_tier
ORDER BY total_customers DESC;

loyalty_tier,total_customers
Platinum,60
Gold,56
Bronze,44
Silver,40


In [0]:
%sql
-- COMMAND ----------

SELECT
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS sales_month,
    ROUND(SUM(gross_sales), 2) AS gross_sales,
    ROUND(SUM(returns_amount), 2) AS total_returns,
    ROUND(SUM(net_revenue), 2) AS net_revenue
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY DATE_FORMAT(transaction_date, 'yyyy-MM')
ORDER BY sales_month;

sales_month,gross_sales,total_returns,net_revenue
2025-01,222416.0,41290.0,160713.5
2025-02,244610.0,50086.5,172090.8
2025-03,413072.0,87870.0,292591.95
2025-04,280183.0,96012.5,169417.7
2025-05,429862.0,145003.5,249144.3
2025-06,287299.0,70774.0,201136.45
2025-07,209809.0,67518.5,124398.45
2025-08,231818.0,53433.0,163500.5
2025-09,379288.0,63784.0,291906.55
2025-10,160617.0,20835.0,129733.0
